# 23 - Spectral Sequences Framework

A **spectral sequence** is the systematic answer to the question: *when can you compute the homology of a complicated space by breaking it into simpler pieces?* Every fibration $F \to E \to B$, every filtered complex, every Postnikov tower gives a spectral sequence. The pages $E_r$ converge to the homology of $E$ via successive approximation.

This notebook covers all five sequence types in `pysurgery.core.spectral_sequences`, plus the stand-alone Adams $E_2$ engine in `pysurgery.core.adams_spectral_sequence`.

## Learning Goals
- Understand spectral sequences as iterated homological approximations
- Use `SerreSpectralSequence` for fibrations (Hopf fibration, path-loop fibration)
- Use `AdamsSpectralSequence` for stable homotopy from Steenrod algebra data
- Use `AtiyahHirzebruchSpectralSequence` (AHSS) for K-theory of CW complexes
- Use `ExactCoupleSpectralSequence` for custom filtered complexes
- Solve extension problems with `solve_extension_problem`
- Interpret `ConvergenceResult`, `SpectralPageSnapshot`, and `ExtensionResult`

## Three-Level View
| Level | Object | Tool |
|---|---|---|
| **Geometric** | Fibration $F \to E \to B$ | `SimplicialComplex`, `CWComplex` |
| **Algebraic** | Bigraded pages $E_r^{p,q}$ with differentials $d_r$ | `SpectralSequence.snapshot(r)` |
| **Result** | $E_\infty$ = associated graded of $H_*(E)$ | `converge()` → `ConvergenceResult` |

## Formal Background

A spectral sequence $(E_r, d_r)_{r \geq 1}$ consists of:
- **Pages**: bigraded abelian groups $E_r^{p,q}$
- **Differentials**: $d_r: E_r^{p,q} \to E_r^{p+r,q-r+1}$ with $d_r^2 = 0$
- **Page transition**: $E_{r+1} = H(E_r, d_r)$

Convergence: $E_\infty^{p,q} \cong F^p H_{p+q}(E) / F^{p+1} H_{p+q}(E)$ for the filtration on $H_*(E)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pysurgery.core.spectral_sequences import (
    SerreSpectralSequence,
    LeraySerreSpectralSequence,
    AdamsSpectralSequence,
    AtiyahHirzebruchSpectralSequence,
    ExactCoupleSpectralSequence,
    solve_extension_problem,
    SpectralPageSnapshot, ConvergenceResult, ExtensionResult,
)
from pysurgery.core.adams_spectral_sequence import (
    adams_e2_page, AdamsE2Page,
    sphere_cohomology_fp,
)
from pysurgery.core.complexes import CWComplex, SimplicialComplex

print("=" * 70)
print("23 - Spectral Sequences Framework: Setup Complete")
print("=" * 70)

## Part 1: The Serre Spectral Sequence

Given a fibration $F \hookrightarrow E \twoheadrightarrow B$ of simply-connected spaces, the Serre spectral sequence has:
$$E_2^{p,q} = H_p(B; H_q(F)) \Rightarrow H_{p+q}(E).$$
The differentials $d_r$ on page $r$ go from $(p,q)$ to $(p-r, q+r-1)$. The spectral sequence converges to the homology of the total space $E$.

**Classic example**: Hopf fibration $S^1 \to S^3 \to S^2$. The $E_2$ page is $H_*(S^2; H_*(S^1))$. A single $d_2$ differential kills everything, and we recover $H_*(S^3)$.

In [ ]:
# Serre SS for the Hopf fibration S¹ → S³ → S²
sss = SerreSpectralSequence(
    base_homology={0: 1, 2: 1},     # H*(S²): Z in degrees 0 and 2
    fibre_homology={0: 1, 1: 1},    # H*(S¹): Z in degrees 0 and 1
    total_dim=3,
)

# Inspect the E₂ page
snap2: SpectralPageSnapshot = sss.snapshot(page=2)
print(f"E₂ page: {len(snap2.nonzero_entries)} nonzero entries")
for entry in snap2.nonzero_entries:
    print(f"  E₂^{{{entry.stem},{entry.filtration}}}: rank={entry.rank}")

# Converge to E∞
conv: ConvergenceResult = sss.converge(max_page=5)
print()
print(f"Converged at page:  {conv.convergence_page}")
print(f"E∞ total homology:  {conv.total_homology}  (should give H*(S³) = Z in 0,3)")
print(f"exact:              {conv.exact}")
print(f"theorem_tag:        {conv.theorem_tag}")

In [ ]:
# Visualise the E₂ and E∞ pages side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, page_num in zip(axes, [2, conv.convergence_page]):
    snap = sss.snapshot(page=page_num)
    if not snap.nonzero_entries:
        ax.set_title(f"E_{page_num} page (empty)"); continue
    max_p = max(e.stem for e in snap.nonzero_entries)
    max_q = max(e.filtration for e in snap.nonzero_entries)
    grid = np.zeros((max_q + 1, max_p + 1))
    for e in snap.nonzero_entries:
        grid[e.filtration, e.stem] = e.rank
    im = ax.imshow(grid, origin="lower", cmap="Blues", vmin=0, vmax=2)
    ax.set_xlabel("p (base degree)"); ax.set_ylabel("q (fiber degree)")
    ax.set_title(f"E_{page_num} page of Hopf Serre SS")
    for (j, i), v in np.ndenumerate(grid):
        if v > 0:
            ax.text(i, j, str(int(v)), ha="center", va="center", fontsize=12)
    plt.colorbar(im, ax=ax)

plt.suptitle("Serre Spectral Sequence: S¹ → S³ → S²")
plt.tight_layout(); plt.show()

## Part 2: Leray-Serre with Twisted Coefficients

When $\pi_1(B) \neq 0$, the fiber action on $H_*(F)$ can be non-trivial. `LeraySerreSpectralSequence` handles this by accepting a representation of $\pi_1(B)$ on the fiber homology.

**Example**: The path-loop fibration $\Omega S^n \to PS^n \to S^n$ (where $PS^n$ is contractible). This recovers the homology of the loop space $\Omega S^n$ inductively.

In [ ]:
# Path-loop fibration ΩS³ → PS³ → S³
# PS³ is contractible: H*(PS³) = Z in degree 0 only
# Using this to compute H*(ΩS³) from Serre SS
lss = LeraySerreSpectralSequence(
    base_homology={0: 1, 3: 1},       # H*(S³)
    total_homology={0: 1},             # H*(PS³) = Z (contractible total space)
    total_dim=0,                       # ΩS³ is not finite-dimensional, set 0 for unbounded
    fiber_dimension_bound=12,          # compute H*(ΩS³) through degree 12
)

conv_loop = lss.converge(max_page=8)
print("H*(ΩS³) through degree 12 (should be Z in all odd degrees):")
for deg, rank in sorted(conv_loop.total_homology.items()):
    if rank > 0:
        print(f"  H_{deg}(ΩS³) = Z^{rank}")
print(f"exact: {conv_loop.exact}")

## Part 3: Adams Spectral Sequence — Stable Homotopy from Cohomology Operations

The Adams spectral sequence computes *stable homotopy groups of spheres* $\pi_n^s(S^0)$ from purely algebraic data: the action of the Steenrod algebra $\mathcal{A}$ on $H^*(X; \mathbb{F}_p)$.

$$E_2^{s,t} = \mathrm{Ext}^{s,t}_{\mathcal{A}}(H^*(X; \mathbb{F}_p), \mathbb{F}_p) \Rightarrow \pi_{t-s}^s(X)_{(p)}.$$

**Two-step workflow:**
1. `adams_e2_page(cohomology, prime)` — computes $\mathrm{Ext}$ via minimal resolution (the $E_2$ page)
2. `AdamsSpectralSequence(e2_page)` — advances through differentials until convergence

In [ ]:
# Step 1: Compute E₂ page for S³ mod 2
H_S3 = sphere_cohomology_fp(n=3, prime=2)
print(f"H*(S³; F₂): generators in degrees {H_S3.generator_degrees}")

e2: AdamsE2Page = adams_e2_page(H_S3, stem_range=12, filtration_range=6)
print()
print(f"E₂ page: {len(e2.nonzero_entries)} nonzero bidegrees, total rank={e2.total_rank}")
print("First 10 nonzero entries (filtration s, stem t-s):")
for entry in e2.nonzero_entries[:10]:
    print(f"  Ext^{{{entry.filtration},{entry.stem}}}: rank={entry.rank}")

In [ ]:
# Step 2: Run the Adams SS to convergence
adams = AdamsSpectralSequence(e2_page=e2, prime=2)
conv_adams = adams.converge(max_page=4)

print(f"Adams SS converged at page: {conv_adams.convergence_page}")
print()
print("Stable homotopy estimate for S³ (mod-2 part):")
for stem, group in sorted(conv_adams.total_homology.items()):
    if group > 0 and stem <= 10:
        print(f"  π_{stem}^s(S³)_{{(2)}} ≈ (Z/2)^{group}")
print(f"exact: {conv_adams.exact}")
print()

# Visualise the Adams chart
fig, ax = plt.subplots(figsize=(12, 5))
for entry in e2.nonzero_entries:
    ax.scatter(entry.stem, entry.filtration, s=60 * entry.rank,
               c="steelblue", alpha=0.7, zorder=3)
ax.set_xlabel("Stem (t - s)"); ax.set_ylabel("Filtration (s)")
ax.set_title("Adams E₂ Chart for S³ at p=2 (dot size = rank)")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Part 4: Atiyah-Hirzebruch Spectral Sequence (K-Theory)

The AHSS converts ordinary cohomology into K-theory. Its $E_2$ page is
$$E_2^{p,q} = H^p(X; K^q(\mathrm{pt})) \Rightarrow K^{p+q}(X),$$
where $K^q(\mathrm{pt}) = \mathbb{Z}$ for $q$ even and $0$ for $q$ odd. The first nontrivial differential is $d_3 = Sq^3$ (Steenrod square composed with integral Bockstein).

**Practical use**: computing $K(X)$ for CW complexes; detecting when a cohomology class *fails* to lift to K-theory (NB 10, Part 6).

In [ ]:
# AHSS for CP³: H*(CP³) = Z[x]/(x⁴), |x|=2
# K(CP³) should be Z⁴ (free of rank 4)
CP3 = CWComplex.complex_projective_space(3)
ahss = AtiyahHirzebruchSpectralSequence(space=CP3, cohomology_theory="K")

snap2_ahss = ahss.snapshot(page=2)
print(f"E₂ page of AHSS for CP³: {len(snap2_ahss.nonzero_entries)} nonzero entries")

conv_ahss = ahss.converge(max_page=5)
print(f"K(CP³) = Z^{conv_ahss.total_homology.get(0, 0)}  (expected Z⁴)")
print(f"Converged at page: {conv_ahss.convergence_page}")
print(f"exact: {conv_ahss.exact}")
print(f"theorem_tag: {conv_ahss.theorem_tag}")
print()

# Compare K-theory with homology for various spaces
test_spaces = {
    "S²":  CWComplex.sphere(2),
    "CP²": CWComplex.complex_projective_space(2),
    "S⁴":  CWComplex.sphere(4),
}
print(f"{'Space':<8} {'H*(X) ranks':<30} {'K(X) rank'}")
print("-" * 55)
for name, X in test_spaces.items():
    ahss_X = AtiyahHirzebruchSpectralSequence(space=X, cohomology_theory="K")
    kX = ahss_X.converge(max_page=5)
    betti = [X.betti_number(k) for k in range(5) if X.betti_number(k) > 0]
    print(f"{name:<8} {str(betti):<30} {kX.total_homology.get(0, 0)}")

## Part 5: Exact Couple Framework

When you have a custom filtered complex (not a fibration), `ExactCoupleSpectralSequence` accepts the exact couple $(D, E, i, j, k)$ directly.

An **exact couple** is a pair of abelian groups with maps $D \xrightarrow{i} D \xrightarrow{j} E \xrightarrow{k} D$ forming a long exact sequence. The derived couple $(D', E', i', j', k')$ has $E' = H(E, jk)$, and iterating this gives the spectral sequence.

This is the universal algebraic framework — every spectral sequence arises from an exact couple.

In [ ]:
# Example: exact couple from the long exact sequence of a CW pair (X, A)
# where X = T² and A = S¹ (the equator circle)
T2 = SimplicialComplex.torus()
S1 = SimplicialComplex.circle()

# Build the exact couple from skeletal filtration of T²
# E₁^{p,q} = H_{p+q}(X_p, X_{p-1}) = cellular chain groups
try:
    ec = ExactCoupleSpectralSequence.from_cw_filtration(T2, max_page=4)
    conv_ec = ec.converge(max_page=4)
    print("Exact couple SS for T² (from CW filtration):")
    print(f"  E∞ total homology: {conv_ec.total_homology}")
    print(f"  Converged at page: {conv_ec.convergence_page}")
    print(f"  exact: {conv_ec.exact}")
except AttributeError:
    # Fallback: construct manually
    print("Using manual exact couple construction:")
    ec = ExactCoupleSpectralSequence(
        D_maps=[T2.boundary_matrix(k).toarray() for k in range(1, 4)],
        E_initial=[T2.cellular_chain_group(k) for k in range(4)],
    )
    conv_ec = ec.converge(max_page=4)
    print(f"  E∞ total homology: {conv_ec.total_homology}")

## Part 6: Solving Extension Problems

The $E_\infty$ page gives only the *associated graded* of $H_*(E)$. The extension problem asks: if $E_\infty^{p,q} = \mathbb{Z}/2$ and $E_\infty^{p+1,q-1} = \mathbb{Z}/2$, is $H_{p+q}(E) = \mathbb{Z}/4$ or $\mathbb{Z}/2 \oplus \mathbb{Z}/2$?

`solve_extension_problem` uses secondary cohomology operations (Massey products, Steenrod operations on $E_\infty$) to resolve this ambiguity.

In [ ]:
# Extension problem: two Z/2 factors in consecutive filtration degrees
result: ExtensionResult = solve_extension_problem(
    filtration_quotients=[2, 2],   # two Z/2 summands from E∞
    degree=1,
    secondary_op="Sq1",            # first Steenrod square as the resolver
)

print("Extension problem: 0 → Z/2 → G → Z/2 → 0")
print(f"  resolved_group: {result.resolved_group}")
print(f"  ambiguous:      {result.ambiguous}")
print(f"  resolution_op:  {result.resolution_op}")
print(f"  exact:          {result.exact}")
print()

# Another case: Z/3 extension (resolved by Steenrod operations at p=3)
result_3 = solve_extension_problem(
    filtration_quotients=[3, 3],
    degree=1,
    secondary_op="beta",           # Bockstein at p=3
)
print("Extension problem: 0 → Z/3 → G → Z/3 → 0")
print(f"  resolved_group: {result_3.resolved_group}")
print(f"  ambiguous:      {result_3.ambiguous}")

## Summary Checklist

- [x] Understood spectral sequences as iterated homological approximations to $H_*(E)$
- [x] Used `SerreSpectralSequence` for the Hopf fibration and read `SpectralPageSnapshot`
- [x] Used `LeraySerreSpectralSequence` to compute $H_*(\Omega S^3)$
- [x] Computed the Adams $E_2$ page via `adams_e2_page` and the Steenrod algebra
- [x] Ran `AdamsSpectralSequence.converge()` to get stable homotopy estimates
- [x] Used `AtiyahHirzebruchSpectralSequence` to compute $K(CP^3)$
- [x] Used `ExactCoupleSpectralSequence` for a custom filtration
- [x] Solved an extension problem with `solve_extension_problem`

## Next Steps
- **NB 29**: Rational surgery uses the Serre SS for $L$-group localization
- **NB 31**: Higher homotopy groups combine the Adams SS output with Sullivan model data
- **NB 34**: The capstone uses the AHSS for the K-theory step of the surgery pipeline